# Milestone - Unit 14: the full system -- Twelve units built twelve kinds of recommender. This milestone runs **the** recommender: one system, end to end, assembled from the regenerated model cache. Retrieval is an equal-weight **8-path pool** (popularity, lexical/BM25, item-item, MF, semantic, the two-tower **served through an HNSW index**, the warm feature tower and the Unit-10 hybrid), blended by calibrated union, then **re-scored by an 8-path learned reranker** trained with Unit 11's leakage-safe recipe. The question is not "does the full stack win?" but "**what earns its place?**" -- so the milestone reports the system next to its ablations, its beyond-accuracy profile and the Part-1 build. The measured story: the full stack scores hit@10 **0.356**, a little *below* the fixed 8-path blend's **0.362** (the reranker dilutes, as Unit 11 found); no system dominates the Part-1 build's reach (coverage **0.333** vs about 0.20); and every leave-one-out row moves hit by at most about 0.01, in both directions. The deliverable is the **hit <-> reach <-> exposure frontier** and a defended choice on it. Everything is fixed-seed and deterministic, on the frozen `val` split only.

## 1. Load the model cache and assemble the system

`build_models.py` fits the expensive models once (MF, the two-tower, the warm and hard-negative feature towers, the two rerankers and SASRec), rebuilds the HNSW index, and writes their artifacts to `recsys/data/generated/models/` -- regenerated on every CI run, exactly like the data. `load_cached_models` refuses a cache built from different data. `FullSystem.from_cache` **loads** every expensive model and refits only the cheap paths (popularity, lexical, item-item, semantic, and the hybrid that reuses the loaded two-tower) -- seconds, not minutes. This notebook performs **no** expensive fit.

In [ ]:
from time import perf_counter

import numpy as np
import pandas as pd
from bookrec import FullSystem, load_book_data, load_cached_models, reader_hits
from bookrec.system import check_readouts

K = 10
notebook_start = perf_counter()

data = load_book_data()
cache = load_cached_models(data_dir=data.data_dir)
system = FullSystem.from_cache(cache, data)
assemble_seconds = perf_counter() - notebook_start
assert len(data.cohort) == 500

for name in ("reranker-u11", "reranker-full8"):
    meta = cache.meta(name)
    # Both cached rerankers carry Unit 11's leakage-safe recipe: the latest 25% of each reader's
    # train positives are the labels, the earlier 75% the profile every feature is computed from.
    assert meta["recipe"] == "u11-holdout-75-25" and meta["profile_frac"] == 0.75, name
assert len(cache.meta("reranker-full8")["path_names"]) == 8

print(f"cache loaded + system assembled (cheap paths refit, expensive ones loaded): {assemble_seconds:.1f}s")
{
    "cached models": sorted(cache.envelopes),
    "val cohort": len(data.cohort),
    "full-stack paths": list(system.full8),
    "reranker features": cache.meta("reranker-full8")["n_features"],
}

## 2. The cache is faithful: every loaded model reproduces its unit's pin

The required check (`tests/test_unit14.py`) verifies the cache in depth -- schemas, a refit of the cheap models, a 20-reader top-10 oracle, the reranker's leakage provenance. Here we re-show the key pins: each loaded model's `val` hit@10 matches the number its own unit measured (within 0.002). The HNSW index is checked against the exact two-tower it approximates (recall@10), the warm feature tower still surfaces about 113 books nobody has read, and SASRec is scored on the **session log** only (design §6).

In [ ]:
pins_start = perf_counter()
PINS = {"mf": 0.276, "two-tower (exact)": 0.340, "two-tower via HNSW": 0.340, "feature tower": 0.298}
served = {
    "mf": system.top_lists(system.full8["mf"]),
    "two-tower (exact)": system.top_lists(system.full8_exact["two-tower"]),
    "two-tower via HNSW": system.top_lists(system.full8["two-tower"]),
    "feature tower": system.top_lists(system.full8["feature-tower"]),
    "reranker (Unit 11, 6 paths)": system.top_lists(system.reranker_u11),
}
PINS["reranker (Unit 11, 6 paths)"] = 0.346
pin_rows = []
for name, lists in served.items():
    hit = float(np.mean([1.0 if set(top) & data.relevant[r] else 0.0 for r, top in lists.items()]))
    assert abs(hit - PINS[name]) <= 0.002, (name, hit)
    pin_rows.append({"cached model": name, "val hit@10": round(hit, 4), "unit pin": PINS[name]})

# HNSW serves (almost) exactly what the exact two-tower would: recall@10 of the ANN lists.
exact, approx = served["two-tower (exact)"], served["two-tower via HNSW"]
ann_recall = float(np.mean([len(set(exact[r]) & set(approx[r])) / K for r in data.cohort]))
assert ann_recall >= 0.995, ann_recall

# The warm feature tower's cold reach: zero-train books surfaced over every warm reader it serves.
counts = system.paths["popularity"].counts                 # {book: train-positive count}
zero_train = set(data.catalog_ids) - set(counts)           # books with no train positive at all
feature_tower = system.paths["feature-tower"]
cold_readers = set(data.cold_readers)
surfaced = set()
for reader in feature_tower.artifact()["reader_ids"]:
    if reader not in cold_readers:
        top = feature_tower.retrieve(reader, {"seen": data.seen_by.get(reader, set())}, K)
        surfaced |= {c.item_id for c in top} & zero_train
assert abs(len(surfaced) - 113) <= 3, len(surfaced)

# SASRec on the session log (its own cohort of 1,419 readers), never on the main log.
session_hits = reader_hits(system.sasrec, data.sessions_path, catalog_ids=data.catalog_ids, k=K)
sasrec_hit = float(np.mean(list(session_hits.values())))
assert len(session_hits) == 1419 and abs(sasrec_hit - 0.4228) <= 0.002, (len(session_hits), sasrec_hit)
pin_rows.append({"cached model": "SASRec (session log)", "val hit@10": round(sasrec_hit, 4), "unit pin": 0.4228})

print(f"pins re-shown: {perf_counter() - pins_start:.1f}s | HNSW recall@10 vs exact: {ann_recall:.4f} | "
      f"feature tower surfaces {len(surfaced)} of {len(zero_train)} zero-train books")
pd.DataFrame(pin_rows)

## 3. The assembled system, the fixed blend and the Part-1 build -- recomputed live

Three systems are scored here from the loaded models, every cohort reader's top-10 on `val`:
- the **full stack** -- the 8-path pool (two-tower via HNSW), re-scored by the 8-path learned reranker (21 features: 8 calibrated scores, 8 presence flags, the path count and 4 content features);
- the **fixed 8-path blend** -- the same pool with equal weights and no reranker;
- the **Part-1 build** -- Unit 6's blend of popularity, lexical, item-item and MF (pool 30, weights 0.25 / 0.5 / 1 / 1), the system Checkpoint A froze.

The best single path, the **Unit-10 hybrid** (BM25 + two-tower), is scored alongside as the reference every combination has to beat. Each readout is pinned (hit and coverage within 0.003; Part-1 within 0.002).

In [ ]:
live_start = perf_counter()
live = {
    "full stack": system.evaluate(system.full_stack),
    "fixed 8-path blend": system.evaluate(system.fixed_blend()),
    "best single path (U10 hybrid)": system.evaluate(system.best_single_path),
    "Part-1 build (U6 blend)": system.evaluate(system.part1_build()),
}
live_seconds = perf_counter() - live_start

LIVE_PINS = {  # name: (hit@10, coverage, tolerance)
    "full stack": (0.356, 0.203, 0.003),
    "fixed 8-path blend": (0.362, 0.206, 0.003),
    "best single path (U10 hybrid)": (0.362, 0.192, 0.003),
    "Part-1 build (U6 blend)": (0.306, 0.333, 0.002),
}
for name, (hit, coverage, tol) in LIVE_PINS.items():
    assert abs(live[name]["hit"] - hit) <= tol, (name, live[name]["hit"])
    assert abs(live[name]["coverage"] - coverage) <= tol, (name, live[name]["coverage"])

print(f"four systems x {len(data.cohort)} readers: {live_seconds:.1f}s")
pd.DataFrame([
    {"system": name, "hit@10": round(m["hit"], 4), "coverage": round(m["coverage"], 4),
     "hit vs Part-1": round(m["hit"] - live["Part-1 build (U6 blend)"]["hit"], 4),
     "coverage vs Part-1": round(m["coverage"] - live["Part-1 build (U6 blend)"]["coverage"], 4)}
    for name, m in live.items()
])

The full stack beats the Part-1 build on hit@10 by 0.05 but gives up about **0.13 of coverage** -- it shows about a fifth of the catalog where Part-1 shows a third. And it does not beat the simpler combinations: the fixed blend and the hybrid both score 0.362. The learned reranker, trained on 8 paths, *dilutes* the pool's best signal rather than sharpening it -- the same finding as Unit 11, now at full scale.

## 4. The ablation table: what earns its place?

The build computes every ablation row from the freshly built cache and writes them to `readouts.json` (regenerated on every CI run, so these values are still verified each time). Two kinds of rows, each with its own **declared baseline**:
- **stage rows** -- remove a stage from the full stack (the reranker; the HNSW index, serving the exact two-tower instead). Deltas are vs the **full stack**.
- **path rows** -- drop one path from the **fixed 8-path blend** (leave-one-out). These are **not reranked**: retraining the reranker without each path would be eight more expensive fits. Deltas are vs the **fixed blend**.

`check_readouts` asserts the structure: exactly these 10 rows, every value finite and in range, and each signed delta equal to *row minus its declared baseline*. The milestone then re-derives those deltas itself and pins each row to its pre-implementation probe value (within 0.003). A pin fixes the **measured** sign for any row that moved by more than 0.003 -- it is a reproducibility check, not a claim that the full stack must win.

In [ ]:
readouts = cache.readouts
check_readouts(readouts)
assert readouts["split"] == "val" and readouts["k"] == K and readouts["cohort_size"] == len(data.cohort)

# The file was built from THIS cache: its baselines equal the live recomputation above.
baselines = readouts["baselines"]
for key, name in (("full-stack", "full stack"), ("fixed-blend", "fixed 8-path blend")):
    for metric in ("hit", "coverage", "exposure_gini"):
        assert abs(baselines[key][metric] - live[name][metric]) <= 1e-9, (key, metric)

PROBE_DELTAS = {  # row: (d_hit, d_coverage), measured by the plan's independent probe
    "no-reranker": (+0.006, +0.0035),
    "exact-two-tower": (-0.002, 0.000),
    "minus-popularity": (+0.008, +0.014),
    "minus-lexical": (0.000, -0.031),
    "minus-item-item": (-0.008, -0.014),
    "minus-mf": (-0.008, +0.0315),
    "minus-semantic": (-0.004, -0.0205),
    "minus-two-tower": (-0.010, +0.015),
    "minus-feature-tower": (+0.002, +0.0015),
    "minus-hybrid": (-0.006, +0.0045),
}
ablation = readouts["ablation"]
assert [row["row"] for row in ablation] == list(PROBE_DELTAS)
ablation_rows = []
for row in ablation:
    base = baselines[row["baseline"]]
    d_hit = row["hit"] - base["hit"]                 # recomputed from the declared baseline
    d_coverage = row["coverage"] - base["coverage"]
    assert abs(d_hit - row["d_hit"]) <= 1e-9 and abs(d_coverage - row["d_coverage"]) <= 1e-9, row["row"]
    probe_hit, probe_coverage = PROBE_DELTAS[row["row"]]
    assert abs(d_hit - probe_hit) <= 0.003, (row["row"], d_hit)
    assert abs(d_coverage - probe_coverage) <= 0.003, (row["row"], d_coverage)
    assert row["kind"] == ("stage" if row["baseline"] == "full-stack" else "path")
    ablation_rows.append({
        "row": row["row"], "kind": row["kind"], "vs": row["baseline"],
        "hit@10": round(row["hit"], 4), "coverage": round(row["coverage"], 4),
        "d hit": round(d_hit, 4), "d coverage": round(d_coverage, 4),
    })

baseline_rows = [
    {"row": key, "kind": "baseline", "vs": "", "hit@10": round(m["hit"], 4),
     "coverage": round(m["coverage"], 4), "d hit": 0.0, "d coverage": 0.0}
    for key, m in baselines.items()
]
pd.DataFrame(baseline_rows + ablation_rows)

**Reading the ablation honestly.** Every row moves hit@10 by at most 0.01 -- most of the pool is redundant for *accuracy*. The rows differ in what they do to **reach**:
- Removing the **reranker** *raises* hit (+0.006) -- the stage costs accuracy here. Serving the **exact** two-tower instead of HNSW changes almost nothing (-0.002 hit, the same coverage): the index is a speed-up, not an accuracy trade.
- Dropping **popularity** *raises* both hit (+0.008) and coverage (+0.014) -- with seven personalised paths, the "everyone gets the bestsellers" voice only crowds the list.
- Dropping **lexical** or **semantic** costs little hit but a lot of **coverage** (-0.031, -0.021): the content paths are what reach the long tail.
- Dropping **MF** or the **two-tower** costs hit (-0.008, -0.010) but *raises* coverage (+0.032, +0.015): the dense collaborative paths buy accuracy by concentrating on the head.
- The **feature tower** is nearly neutral on the warm cohort (+0.002 when dropped) -- its job is the cold books no warm reader's hit@10 can see.

No single path is pure gain or pure loss. Which ones "earn their place" depends on whether you weigh hit, reach, or cold-start.

## 5. The beyond-accuracy table

Accuracy is one column. For the three systems the capstone compares -- the **full system**, the **best single path** and the **Part-1 build** -- the readout adds five beyond-accuracy columns (Unit 6 and Unit 13):
- **coverage** -- the fraction of the 2000-book catalog shown to at least one reader;
- **head-share** -- the fraction of all slots that go to the top-10% most-read books;
- **exposure Gini** -- the Gini of per-book slot counts over the **full catalog** (never-shown books count as 0);
- **novelty** -- the mean self-information of the recommended books (higher = less popular books);
- **diversity** -- the mean intra-list dissimilarity (1 - item-item cosine) within a reader's top-10.

Each row is pinned to its build readout, and the full-system and Part-1 rows must equal the live recomputation in section 3.

In [ ]:
beyond = {row["system"]: row for row in readouts["beyond_accuracy"]}
columns = readouts["beyond_accuracy_columns"]
assert list(beyond) == ["full-system", "best-single-path", "part1-build"]
assert columns == ["coverage", "head_share", "exposure_gini", "novelty", "diversity"]

# The rows are the live systems of section 3, all columns (the file was built from this cache).
for key, name in (("full-system", "full stack"), ("best-single-path", "best single path (U10 hybrid)"),
                  ("part1-build", "Part-1 build (U6 blend)")):
    for column in ("hit", *columns):
        assert abs(beyond[key][column] - live[name][column]) <= 1e-9, (key, column)

BEYOND_PINS = {  # system: (coverage, head-share, exposure Gini, novelty, diversity)
    "full-system": (0.2025, 0.8466, 0.9131, 8.568, 0.8566),
    "best-single-path": (0.1920, 0.8616, 0.9220, 8.483, 0.8501),
    "part1-build": (0.3330, 0.7930, 0.9080, 8.492, 0.8711),
}
for key, pins in BEYOND_PINS.items():
    for column, pin in zip(columns, pins, strict=True):
        tol = 0.02 if column == "novelty" else 0.003    # novelty is in bits, not a [0, 1] share
        assert abs(beyond[key][column] - pin) <= tol, (key, column, beyond[key][column])

pd.DataFrame([
    {"system": key, "hit@10": round(row["hit"], 4), "coverage": round(row["coverage"], 4),
     "head-share": round(row["head_share"], 4), "exposure Gini": round(row["exposure_gini"], 4),
     "novelty": round(row["novelty"], 3), "diversity": round(row["diversity"], 4)}
    for key, row in beyond.items()
])

The beyond-accuracy table tells the same story from the other side. The Part-1 build is the **least accurate** system but the one with the widest reach: the highest coverage (0.333), the lowest head-share (0.793), the lowest exposure Gini (0.908) and the most diverse lists. The full system buys +0.05 hit with a narrower, more head-heavy catalog slice; its novelty is the highest of the three, so the *few* tail books it does show are deeper in the tail. The best single path is the most accurate and the most concentrated. No row wins every column.

## 6. The frontier, and what the capstone defends

In [ ]:
frontier = []
for key, row in beyond.items():
    frontier.append({
        "system": key,
        "hit@10": round(row["hit"], 4),
        "coverage": round(row["coverage"], 4),
        "exposure Gini": round(row["exposure_gini"], 4),
        "beats Part-1 on hit": row["hit"] > beyond["part1-build"]["hit"],
        "beats Part-1 on reach": row["coverage"] > beyond["part1-build"]["coverage"],
    })
# No assembled system dominates the Part-1 build on reach -- the trade is real, and measured.
assert not any(r["beats Part-1 on reach"] for r in frontier if r["system"] != "part1-build")

total_seconds = perf_counter() - notebook_start
print(f"whole milestone: {total_seconds:.1f}s")
pd.DataFrame(frontier)

**The honest headline.** The full recommender works -- every cached model reproduces its unit's number, and the assembled stack scores 0.356 hit@10 against Part-1's 0.306. But "more machinery" is not "better on every axis". The learned reranker scores slightly *below* the plain fixed blend it re-ranks (0.356 vs 0.362); the 8-path leave-one-out moves hit by at most 0.01 per path, with the content paths carrying **reach** and the dense paths carrying **accuracy**; and no system matches the Part-1 build's coverage. A real team would pick a point on this **hit <-> reach <-> exposure frontier** -- and could defend more than one.

**The bridge.** The capstone (`project-14-bookrec`) asks you to do exactly that: choose your retrieval paths, build the pool and blend, add (or decide against) the reranker, produce your own ablation and beyond-accuracy tables against the Part-1 benchmark, and write the design memo that defends your point on the frontier -- accuracy vs reach vs exposure, cold start, and cost.